In [7]:
import yfinance as yf
import pandas as pd
import os
import time

In [8]:
START = "2011-01-01"
END   = "2025-12-31"

TICKERS = {
    # ── Original 20 ───────────────────────────────────────────────
    "AAPL":  "AAPL",   # Apple
    "MSFT":  "MSFT",   # Microsoft
    "NVDA":  "NVDA",   # NVIDIA
    "AMZN":  "AMZN",   # Amazon
    "JPM":   "JPM",    # JPMorgan Chase
    "JNJ":   "JNJ",    # Johnson & Johnson
    "XOM":   "XOM",    # ExxonMobil
    "TSLA":  "TSLA",   # Tesla
    "NFLX":  "NFLX",   # Netflix
    "V":     "V",      # Visa
    "GOOGL": "GOOGL",  # Alphabet
    "META":  "META",   # Meta  (IPO May 2012 — effective start ~2013 after OLS warmup)
    "BRK_B": "BRK-B",  # Berkshire Hathaway B
    "UNH":   "UNH",    # UnitedHealth
    "MA":    "MA",     # Mastercard
    "HD":    "HD",     # Home Depot
    "PG":    "PG",     # Procter & Gamble
    "COST":  "COST",   # Costco
    "DIS":   "DIS",    # Disney
    "INTC":  "INTC",   # Intel

    # ── New 30 (all listed before 2011, no survivorship bias) ─────
    # Financials
    "BAC":   "BAC",    # Bank of America
    "WFC":   "WFC",    # Wells Fargo
    "GS":    "GS",     # Goldman Sachs
    "AXP":   "AXP",    # American Express
    "BLK":   "BLK",    # BlackRock

    # Healthcare
    "PFE":   "PFE",    # Pfizer
    "MRK":   "MRK",    # Merck
    "ABT":   "ABT",    # Abbott Laboratories
    "TMO":   "TMO",    # Thermo Fisher Scientific
    "LLY":   "LLY",    # Eli Lilly

    # Technology
    "ORCL":  "ORCL",   # Oracle
    "CSCO":  "CSCO",   # Cisco
    "QCOM":  "QCOM",   # Qualcomm
    "TXN":   "TXN",    # Texas Instruments
    "ADBE":  "ADBE",   # Adobe
    "CRM":   "CRM",    # Salesforce (IPO 2004)
    "AVGO":  "AVGO",   # Broadcom (IPO 2009)

    # Consumer
    "WMT":   "WMT",    # Walmart
    "MCD":   "MCD",    # McDonald's
    "NKE":   "NKE",    # Nike
    "SBUX":  "SBUX",   # Starbucks
    "LOW":   "LOW",    # Lowe's

    # Energy
    "CVX":   "CVX",    # Chevron
    "COP":   "COP",    # ConocoPhillips

    # Industrials
    "BA":    "BA",     # Boeing
    "CAT":   "CAT",    # Caterpillar
    "HON":   "HON",    # Honeywell
    "UPS":   "UPS",    # UPS

    # Telecom
    "VZ":    "VZ",     # Verizon
    "T":     "T",      # AT&T
}

ticker_list = tuple(TICKERS.values())
print(f"Total tickers: {len(TICKERS)}")

Total tickers: 50


In [9]:
raw = {}
for name, ticker in TICKERS.items():
    print(f"Downloading {name} ({ticker})...")
    for attempt in range(1, 6):
        try:
            df = yf.download(
                ticker,
                start=START,
                end=END,
                auto_adjust=True,
                progress=False,
                threads=False,
            )
            if not df.empty:
                break
        except Exception as e:
            df = pd.DataFrame()
            print(f"  Error: {e}")
        wait = attempt * 30
        print(f"  Retrying in {wait}s (attempt {attempt}/5)...")
        time.sleep(wait)

    if df.empty:
        print(f"  FAILED: {name}")
        continue

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.index = pd.to_datetime(df.index)
    raw[name] = df
    print(f"  {len(df)} rows  |  {df.index[0].date()} -> {df.index[-1].date()}")
    time.sleep(2)

  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3424 rows  |  2012-05-18 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  2011-01-03 -> 2025-12-30
  3771 rows  |  

In [10]:
# Preview each downloaded DataFrame
for name, df in raw.items():
    print(f"\n--- {name} ---")
    print(df.shape)
    print(df.isna().sum())
    display(df.tail())


--- AAPL ---
(3771, 5)
Price
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


Price,Close,High,Low,Open,Volume
Date,,,,,
2025-12-23,272.105377,272.245261,269.308007,270.586809,29642000
2025-12-24,273.554016,275.172497,271.945536,272.085389,17910600
2025-12-26,273.144409,275.112569,272.604905,273.903708,21521800
2025-12-29,273.504089,274.103504,272.095404,272.435082,23715200
2025-12-30,272.824707,273.823772,272.025467,272.554970,22139600



--- MSFT ---
(3771, 5)
Price
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


Price,Close,High,Low,Open,Volume
Date,,,,,
2025-12-23,485.741333,486.719082,483.636122,483.875596,14683600
2025-12-24,486.908630,488.046049,483.725892,484.573963,5855900
2025-12-26,486.599365,487.008435,484.853350,485.601642,8842200
2025-12-29,485.990753,487.237907,483.077389,483.755834,10893400
2025-12-30,486.369904,488.564875,484.394402,484.823415,13944500



--- NVDA ---
(3771, 5)
Price
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


...

Price,Close,High,Low,Open,Volume
Date,,,,,
2025-12-23,98.812012,99.512246,98.506270,99.305135,4645300
2025-12-24,99.275551,99.679908,98.526002,98.821871,1845500
2025-12-26,99.157196,99.502381,98.575307,99.137467,2856700
2025-12-29,98.309021,99.610866,97.697546,99.127607,4352600
2025-12-30,98.269577,98.792286,98.121639,98.348479,3397900



--- VZ ---
(3771, 5)
Price
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


Price,Close,High,Low,Open,Volume
Date,,,,,
2025-12-23,38.657764,38.928914,38.609346,38.735237,19761800
2025-12-24,39.045116,39.064484,38.628712,38.667448,7949500
2025-12-26,39.200058,39.258162,38.996699,38.996699,11875400
2025-12-29,39.200058,39.413103,39.103221,39.229108,16846700
2025-12-30,39.413101,39.480888,39.141952,39.200056,15581300



--- T ---
(3771, 5)
Price
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


Price,Close,High,Low,Open,Volume
Date,,,,,
2025-12-23,23.926140,23.994614,23.730506,23.750070,20869100
2025-12-24,24.121775,24.170683,23.838105,23.838105,13904100
2025-12-26,24.111992,24.229374,24.053302,24.072866,16070100
2025-12-29,24.229376,24.395666,24.141340,24.141340,30322100
2025-12-30,24.268501,24.317411,24.190247,24.229375,21047900


In [11]:
# Summary statistics on Close prices
for name, df in raw.items():
    print(f"\n--- {name} ---")
    display(df["Close"].describe())


--- AAPL ---


count    3771.000000
mean       82.041422
std        75.103734
min         9.447933
25%        21.597765
50%        41.481041
75%       145.910072
max       285.922455
Name: Close, dtype: float64


--- MSFT ---


count    3771.000000
mean      157.820667
std       145.806549
min        18.272852
25%        36.674103
50%        93.667831
75%       262.643402
max       539.825256
Name: Close, dtype: float64


--- NVDA ---


...

count    3771.000000
mean       92.376061
std        38.084378
min        37.279938
25%        65.767078
50%        80.865898
75%       125.262897
max       190.228745
Name: Close, dtype: float64


--- VZ ---


count    3771.000000
mean       31.507943
std         7.420906
min        15.634316
25%        26.370406
50%        30.871035
75%        38.731867
max        44.310104
Name: Close, dtype: float64


--- T ---


count    3771.000000
mean       14.411518
std         4.159376
min         7.070363
25%        11.502007
50%        14.474769
75%        15.883247
max        28.664915
Name: Close, dtype: float64

In [12]:
os.makedirs("../data/stocks", exist_ok=True)

for name, df in raw.items():
    out = f"../data/stocks/raw_{name.lower()}.csv"
    df.to_csv(out)
    print(f"Saved {out}")

Saved ../data/stocks/raw_aapl.csv
Saved ../data/stocks/raw_msft.csv
Saved ../data/stocks/raw_nvda.csv
Saved ../data/stocks/raw_amzn.csv
Saved ../data/stocks/raw_jpm.csv
Saved ../data/stocks/raw_jnj.csv
Saved ../data/stocks/raw_xom.csv
Saved ../data/stocks/raw_tsla.csv
Saved ../data/stocks/raw_nflx.csv
Saved ../data/stocks/raw_v.csv
Saved ../data/stocks/raw_googl.csv
Saved ../data/stocks/raw_meta.csv
Saved ../data/stocks/raw_brk_b.csv
Saved ../data/stocks/raw_unh.csv
Saved ../data/stocks/raw_ma.csv
Saved ../data/stocks/raw_hd.csv
Saved ../data/stocks/raw_pg.csv
Saved ../data/stocks/raw_cost.csv
Saved ../data/stocks/raw_dis.csv
Saved ../data/stocks/raw_intc.csv
Saved ../data/stocks/raw_bac.csv
Saved ../data/stocks/raw_wfc.csv
Saved ../data/stocks/raw_gs.csv
Saved ../data/stocks/raw_axp.csv
Saved ../data/stocks/raw_blk.csv
Saved ../data/stocks/raw_pfe.csv
Saved ../data/stocks/raw_mrk.csv
Saved ../data/stocks/raw_abt.csv
Saved ../data/stocks/raw_tmo.csv
Saved ../data/stocks/raw_lly.csv
Sav